# Pastela Pharmaceuticals — Pharmacovigilance Risk Analytics
**Data Analyst Portfolio Project — Python, SQL & Tableau**

You are the newest data analyst on Pastela Pharmaceuticals' Data & Analytics team. The internal Pharmacovigilance department wants to move from reacting to adverse-event reports to proactively flagging high-risk active treatments. Your manager, Julian Cross, and your colleague, Priya Anand, have asked you to build the analysis end to end: pull the data with SQL, clean and model it in Python, and export the results for a Tableau dashboard.

Full background, the team roster, the kickoff emails, and the stakeholder interview are in [`docs/pharma_analytics_project_brief.md`](docs/pharma_analytics_project_brief.md) — read that first if you haven't.

This notebook assumes its working directory is the project root (the folder containing `raw/`, `docs/`, and `processed/`).

## PACE stages

This project follows the **PACE** framework used throughout this portfolio:

* **Plan** — define the objective and success criteria
* **Analyze** — extract, clean, explore, and statistically test the data
* **Construct** — build and evaluate the predictive model
* **Execute** — export results, design the Tableau dashboard, and communicate findings

## PACE: Plan

### Task 1a. Define the objective

Read the stakeholder interview in the project brief, then answer:

1. What is the target variable this model predicts?

**Your response:**

2. Who is the end user of this model's output, and what decision will it inform?

**Your response:**

3. What must the model explicitly *not* be used for (per Marcus's regulatory note)?

**Your response:**

4. What accessibility constraint does the Tableau dashboard have to satisfy, and why?

**Your response:**

### Task 1b. Imports and setup

Import the libraries needed across the whole pipeline: `pandas`/`numpy` for data handling, `sqlalchemy` for the SQL step, `matplotlib`/`seaborn` for visualization, `scipy`/`statsmodels` for statistical inference, and `scikit-learn` for modeling.

In [ ]:
# TODO: import pandas, numpy, sqlalchemy (create_engine, text)
# TODO: import matplotlib.pyplot and seaborn for visualization
# TODO: import scipy.stats and statsmodels.api for statistical inference
# TODO: import train_test_split, OneHotEncoder, StandardScaler,
#       ColumnTransformer, Pipeline, LogisticRegression,
#       DecisionTreeClassifier, and the sklearn.metrics you'll need

RANDOM_STATE = 42

## PACE: Analyze

Consider these questions in your PACE Strategy Document while working through this stage:
* What do you do about missing data?
* Are there outliers, and how should you treat them?
* What do the distributions tell you about the problem you're solving?

### Task 2. Extract data with SQL

Pastela stores its pharmacovigilance data in the five source tables documented in [`raw/schema_pharma_analytics_db.sql`](raw/schema_pharma_analytics_db.sql). Rather than pulling every column of every table into memory, connect with Python (`SQLAlchemy`) and write a single filtering `JOIN` query that returns only the columns relevant to the adverse-event risk question — this keeps the heavy lifting in the database engine instead of pandas.

Here, the CSV files in `raw/` are first loaded into a local SQLite database (standing in for the company's data warehouse), so the SQL step below is a real SQL round-trip, not just `read_csv`.

In [ ]:
# TODO: point RAW_DIR at the raw/ folder and PROCESSED_DIR at processed/
# TODO: create a SQLAlchemy engine for a local SQLite file, e.g.
#       processed/pharma_analytics.db
# TODO: loop over the five CSV files in raw/ and load each one into a
#       same-named SQL table with DataFrame.to_sql()


In [ ]:
# TODO: write a SQL query (as a SQLAlchemy text()) that joins
#       treatments -> patients, treatments -> medications, and
#       LEFT JOINs adverse_events (a treatment may have zero events)
# TODO: only SELECT the columns you'll actually need downstream
# TODO: build an adverse_event_flag column with a CASE expression
# TODO: run the query with pd.read_sql() into a DataFrame called df


### Task 3. Data exploration and cleaning

Check dtypes, null counts, and duplicates. Decide what each null actually means before you touch it — a null `severity` here doesn't mean *missing data*, it means *no adverse event occurred*.

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# TODO: fill the 'severity' null values with a category like "None"
#       (a null here means no adverse event, not missing data)
# TODO: fill 'required_hospitalization' nulls with 0 and cast to int
# TODO: check for and drop any exact duplicate rows


### Task 4. Feature engineering

Combine and transform the raw columns into variables a model can actually use: patient age at the time treatment started, BMI, how many concurrent treatments a patient is on (polypharmacy), and treatment duration.

In [ ]:
# TODO: convert birth_date, start_date, end_date to datetime
# TODO: compute age_at_start = (start_date - birth_date) in years
# TODO: compute bmi = weight_kg / (height_cm / 100) ** 2
# TODO: compute polypharmacy_count = number of treatments per patient_id
#       (hint: groupby + transform("count"))
# TODO: compute treatment_duration_days, filling ongoing treatments
#       (end_date is null) with days-since-start instead


### Task 5. Descriptive statistics and exploratory visualization

Get a feel for the distributions before modeling anything, and look for an early visual signal that adverse events aren't randomly distributed across the population.

In [ ]:
df[["age_at_start", "bmi", "dose_mg", "polypharmacy_count", "treatment_duration_days"]].describe()

In [ ]:
# TODO: plot a histogram of age_at_start
# TODO: plot a histogram of bmi
# TODO: plot a boxplot comparing age_at_start for adverse_event_flag == 0 vs 1


In [ ]:
# TODO: group by who_atc_category and compute the mean of adverse_event_flag
#       (this gives the adverse event rate per medication category)
# TODO: sort descending and bar-plot the top 10 categories


### Task 6. Statistical inference

Before trusting a predictive model, confirm with formal statistical tests that the associations we're seeing are unlikely to be chance:
* A **chi-square test of independence** between medication category and adverse-event occurrence (two categorical variables).
* A **t-test** comparing patient age between the event and no-event groups (categorical vs. continuous).
* A **logistic regression** fit with `statsmodels` to get interpretable coefficients, p-values, and confidence intervals for the numeric risk factors.

In [ ]:
# TODO: build a contingency table with pd.crosstab(who_atc_category, adverse_event_flag)
# TODO: run scipy.stats.chi2_contingency() on it and report the p-value


In [ ]:
# TODO: split age_at_start into two Series by adverse_event_flag (0 vs 1)
# TODO: run scipy.stats.ttest_ind() (equal_var=False) and report the result


In [ ]:
# TODO: pick a handful of numeric features (age_at_start, bmi, dose_mg,
#       polypharmacy_count), add a constant with sm.add_constant()
# TODO: fit sm.Logit(y, X) and print the summary (coefficients, p-values, CIs)


**Interpretation:** Which factors came back statistically significant (p < 0.05)? Do the coefficient signs match clinical intuition (e.g., does risk increase with age)?

**Your response:**

## PACE: Construct

Consider these questions while working through this stage: which features are safe to use (no data leakage from post-event columns like `severity`?), how should you handle the class imbalance in `adverse_event_flag`, and how will you decide between the two models?

### Task 7. Prepare features and split the data

Use the numeric features engineered above plus the categorical medication/route/frequency/sex columns. **Do not** include `severity` or `required_hospitalization` as predictors — those are only known *after* an adverse event has already happened, so including them would leak the answer.

In [ ]:
# TODO: define numeric_features and categorical_features lists
#       (exclude severity / required_hospitalization -- data leakage)
# TODO: build X (features) and y (adverse_event_flag)
# TODO: train_test_split with test_size=0.2, stratify=y, random_state=RANDOM_STATE


### Task 8. Train Logistic Regression and Decision Tree models

Build a `ColumnTransformer` that scales the numeric features and one-hot encodes the categorical ones, then train both a Logistic Regression and a Decision Tree classifier through the same preprocessing pipeline so they're evaluated on an equal footing.

In [ ]:
# TODO: build a ColumnTransformer: StandardScaler on numeric_features,
#       OneHotEncoder(handle_unknown="ignore") on categorical_features
# TODO: wrap the preprocessor + LogisticRegression(class_weight="balanced")
#       in a Pipeline named log_reg, and fit it on the training data
# TODO: do the same with a DecisionTreeClassifier(max_depth=5,
#       class_weight="balanced") in a Pipeline named tree_clf


### Task 9. Evaluate and compare the models

Because adverse events are the minority class (~8% of treatments), accuracy alone is misleading — look at precision/recall and ROC-AUC.

In [ ]:
# TODO: for each model, predict on X_test, get predicted probabilities
#       with predict_proba()[:, 1]
# TODO: print a classification_report and the roc_auc_score for each model


In [ ]:
# TODO: plot ROC curves for both models on the same axes
#       (RocCurveDisplay.from_estimator)
# TODO: plot a confusion matrix for at least one model
#       (ConfusionMatrixDisplay.from_estimator)


**Model comparison:** Which model performed better on ROC-AUC? What's the tradeoff between the two (interpretability vs. performance)?

**Your response:**

In [ ]:
# TODO: get the one-hot encoded feature names out of the fitted
#       ColumnTransformer (get_feature_names_out)
# TODO: plot_tree() on the fitted DecisionTreeClassifier, limited to
#       max_depth=2 so it stays readable


### Task 10. Export the scored dataset for Tableau

Consolidate the cleaned data with both models' predicted probabilities into a single file Tableau can connect to directly.

In [ ]:
# TODO: copy df into export_df
# TODO: add predicted_risk_logreg and predicted_risk_tree columns using
#       each pipeline's predict_proba(X)[:, 1]
# TODO: bucket predicted_risk_logreg into a risk_tier column
#       (e.g. Low / Medium / High)
# TODO: write export_df to processed/tableau_ready_predictions.csv


In [ ]:
# TODO (optional): if pantab is installed, use pantab.frame_to_hyper()
#       to also write a native .hyper extract for Tableau -- wrap the
#       import in a try/except ImportError since it's an optional dependency


## PACE: Execute

Consider the PACE Strategy Document to reflect on this stage: how will you present a probabilistic model to a non-technical audience, and what's the one chart Marta and Julian need to see first?

### Task 11. Plan the Tableau dashboard

Connect Tableau to `processed/tableau_ready_predictions.csv` (or the `.hyper` extract) and design a dashboard for Helena's team. Sketch out:

1. **KPIs** — what 2-3 numbers should be visible at a glance?

**Your response:**

2. **Calculated fields** — what derived fields will you need beyond what's in the export (e.g., a risk-tier bucket)?

**Your response:**

3. **Filters** — what should a Pharmacovigilance analyst be able to slice by?

**Your response:**

4. **Accessibility** — given Helena's visual impairment, how will you avoid relying on color alone?

**Your response:**

### Task 12. Executive summary

Write a short executive summary for Marta Fields and Julian Cross covering:

1. **Summary of tasks completed**

**Your response:**

2. **Results of the data/variable assessment**

**Your response:**

3. **Recommended next steps**

**Your response:**

## Appendix — required libraries

| Category | Library | Why |
|---|---|---|
| SQL connectivity | `SQLAlchemy` | Connects Python to the (local SQLite / any relational) database and runs the extraction query |
| Data manipulation | `pandas` | DataFrames, cleaning, joins, grouping |
| Numerical computation | `numpy` | Vectorized math, `NaN` handling |
| Statistics | `scipy`, `statsmodels` | Hypothesis tests, inferential logistic regression |
| Machine learning | `scikit-learn` | Preprocessing, LogisticRegression, DecisionTreeClassifier, evaluation metrics |
| Visualization | `matplotlib`, `seaborn` | Distributions, boxplots, ROC curves |
| Tableau integration | `pantab` | Writes a native `.hyper` extract directly from a pandas DataFrame (optional; the CSV export works without it) |
| Tableau integration | `tableauserverclient` | Automates publishing the extract to Tableau Server/Cloud (optional, not exercised in this notebook) |